**INI PASTINYA FIX**

In [ ]:
import cv2
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
import nest_asyncio
import math

# Mengaktifkan nest_asyncio agar Gradio berjalan lancar di server Colab
nest_asyncio.apply()

# =========================================================================
# REFERENSI DIMENSI FISIK KOIN BANK INDONESIA (dalam satuan mm & rasio)
# Sumber: Spesifikasi resmi Bank Indonesia
# =========================================================================
#
# Fitur yang digunakan untuk klasifikasi:
#   - Diameter_mm   : diameter fisik koin (mm)
#   - Area_mm2      : luas area koin (mm²), dihitung dari diameter: π*(d/2)²
#   - Perimeter_mm  : keliling koin (mm), dihitung dari diameter: π*d
#   - Circularity   : rasio kebundaran (tanpa satuan), idealnya mendekati 1.0
#
REFERENSI_KOIN = {

    "Rp100": {
        "Diameter_mm" : 23.0,                          # Ø 23.0 mm (Aluminium Kakatua resmi)
        "Area_mm2"    : math.pi * (23.0 / 2) ** 2,    # ≈ 415.48 mm²
        "Perimeter_mm": math.pi * 23.0,               # ≈ 72.26 mm
        "Circularity" : 0.90,
    },

    "Rp200": {
        "Diameter_mm" : 25.0,                          # Ø 25.0 mm (Aluminium Jalak Bali resmi)
        "Area_mm2"    : math.pi * (25.0 / 2) ** 2,    # ≈ 490.87 mm²
        "Perimeter_mm": math.pi * 25.0,               # ≈ 78.54 mm
        "Circularity" : 0.90,
    },

    "Rp500": {
        "Diameter_mm" : 27.0,                          # Ø 27.0 mm (Bunga Melati resmi)
        "Area_mm2"    : math.pi * (27.0 / 2) ** 2,    # ≈ 572.56 mm²
        "Perimeter_mm": math.pi * 27.0,               # ≈ 84.82 mm
        "Circularity" : 0.90,
    },

    "Rp1000": {
        "Diameter_mm" : 24.15,                         # Ø 24.15 mm (Baja berlapis Nikel Angklung resmi)
        "Area_mm2"    : math.pi * (24.15 / 2) ** 2,   # ≈ 458.07 mm²
        "Perimeter_mm": math.pi * 24.15,              # ≈ 75.87 mm
        "Circularity" : 0.90,
    },

}

# Bobot per fitur dalam weighted scoring
# Diameter mendapat bobot tertinggi karena paling diskriminatif antar nominal
BOBOT_FITUR = {

    "Diameter_mm" : 0.60,   # fitur paling penting — beda antar nominal jelas
    "Area_mm2"    : 0.20,   # memperkuat diameter
    "Perimeter_mm": 0.05,   # redundan dengan diameter, bobot kecil
    "Circularity" : 0.15,   # membantu menyaring bentuk non-bulat

}

# =========================================================================
# SIGMA (TOLERANSI) PER FITUR — dipakai oleh Gaussian Similarity
# =========================================================================
# Gaussian similarity: score = exp(-0.5 * ((x - ref) / sigma)^2)
# Semakin kecil sigma → semakin ketat toleransi → lebih diskriminatif
#
# Referensi perbedaan diameter antar koin BI:
#   Rp100=22mm, Rp200=23mm, Rp1000=24mm, Rp500=27mm
#   Selisih terkecil = 1 mm  →  sigma Diameter = 1.0 mm
#
SIGMA_FITUR = {
    "Diameter_mm" : 1.0,    # toleransi ±1 mm sangat ketat, ideal untuk BI
    "Area_mm2"    : 25.0,   # toleransi ±25 mm² (≈ selisih area Rp100 vs Rp200 ~35mm²)
    "Perimeter_mm": 3.2,    # toleransi ±3.2 mm (≈ π × 1mm diameter)
    "Circularity" : 0.05,   # toleransi ±0.05 untuk circularity
}

# Threshold minimum confidence — di bawah ini dianggap "Tidak Dikenal"
KONFIDENSI_MIN = 0.40

# =========================================================================
# FUNGSI BANTUAN (HELPER FUNCTIONS)
# =========================================================================
def hitung_hu_moments_log(contour):
    moments = cv2.moments(contour)
    hu = cv2.HuMoments(moments).flatten()
    for i in range(len(hu)):
        if hu[i] != 0:
            hu[i] = -1 * math.copysign(1.0, hu[i]) * math.log10(abs(hu[i]))
        else:
            hu[i] = 0
    return hu

def ekstrak_matriks_pusat(citra_gray):
    h, w = citra_gray.shape
    cy, cx = h // 2, w // 2
    y1, y2 = max(0, cy - 2), min(h, cy + 3)
    x1, x2 = max(0, cx - 2), min(w, cx + 3)
    patch = citra_gray[y1:y2, x1:x2]
    pad_t, pad_b = max(0, 2 - cy), max(0, (cy + 3) - h)
    pad_l, pad_r = max(0, 2 - cx), max(0, (cx + 3) - w)
    patch = np.pad(patch, ((pad_t, pad_b), (pad_l, pad_r)), mode='edge')
    return patch[:5, :5], int(citra_gray[cy, cx])

def generate_stage_visuals(img, stage_num, stage_name):
    """Menghasilkan Histogram dan Log Matriks 5x5 untuk setiap tahapan"""
    # 1. Konversi ke Grayscale untuk analisis jika citra masih RGB
    if len(img.shape) == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray_img = img.copy()

    # 2. Buat Histogram
    fig = plt.figure(figsize=(5, 3))
    plt.hist(gray_img.ravel(), 256, [0, 256], color='gray')
    plt.title(f"Histogram - {stage_name}")
    plt.tight_layout()

    # 3. Buat Log Matriks 5x5
    patch, center_val = ekstrak_matriks_pusat(gray_img)
    mean_val = np.mean(patch)
    min_val = np.min(patch)
    max_val = np.max(patch)

    mat_str = ""
    for row in patch:
        mat_str += " ".join([f"{v:3}" for v in row]) + "\n"

    log_str = (
        f"======================\n"
        f"TAHAP {stage_num}\n"
        f"{stage_name.upper()}\n"
        f"Center Pixel : {center_val}\n"
        f"Matrix 5x5\n"
        f"{mat_str}"
        f"Mean : {mean_val:.2f}\n"
        f"Min : {min_val}\n"
        f"Max : {max_val}\n"
        f"======================"
    )

    return fig, log_str

# =========================================================================
# PIPELINE UTAMA
# =========================================================================
def pipeline_koin_stabil(gambar_input, val_bright, val_contrast, val_blur, val_t_block, val_t_c, val_ppm, val_auto_ppm):
    # Fallback jika gambar kosong
    if gambar_input is None:
        blank_img = np.zeros((100,100), dtype=np.uint8)
        blank_fig = plt.figure(); plt.close(blank_fig)
        blank_outputs = [(blank_img, blank_fig, "")] * 10
        flat_blank = [item for sublist in blank_outputs for item in sublist]
        return tuple(flat_blank + [[], "", blank_img, "⚠️ Unggah gambar terlebih dahulu."])

    if len(gambar_input.shape) == 3 and gambar_input.shape[2] == 4:
        img_raw = gambar_input[:, :, :3].copy()
    else:
        img_raw = gambar_input.copy()

    outputs = [] # Menyimpan sekumpulan (Image, Histogram, Matrix_Log)

    # ---------------------------------------------------------
    # TAHAP 1: SCALING
    # ---------------------------------------------------------
    lebar_target = 1000
    rasio = lebar_target / float(img_raw.shape[1])
    tinggi_target = int(img_raw.shape[0] * rasio)
    img_scaled = cv2.resize(img_raw, (lebar_target, tinggi_target), interpolation=cv2.INTER_AREA)

    fig1, log1 = generate_stage_visuals(img_scaled, 1, "Scaling")
    outputs.extend([img_scaled, fig1, log1])

    # ---------------------------------------------------------
    # TAHAP 2: SHADOW REMOVAL
    # ---------------------------------------------------------
    img_bgr = cv2.cvtColor(img_scaled, cv2.COLOR_RGB2BGR)
    kanals = cv2.split(img_bgr)
    kanals_bersih = []
    kernel_bg = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))

    for kanal in kanals:
        bg_estimasi = cv2.morphologyEx(kanal, cv2.MORPH_CLOSE, kernel_bg)
        bg_estimasi = cv2.GaussianBlur(bg_estimasi, (21, 21), 0)
        kanal_compensated = cv2.divide(kanal, bg_estimasi, scale=255)
        kanals_bersih.append(kanal_compensated)

    img_shadow_removed = cv2.merge(kanals_bersih)
    img_shadow_rgb = cv2.cvtColor(img_shadow_removed, cv2.COLOR_BGR2RGB)

    fig2, log2 = generate_stage_visuals(img_shadow_rgb, 2, "Shadow Removal")
    outputs.extend([img_shadow_rgb, fig2, log2])

    # ---------------------------------------------------------
    # TAHAP 3: GRAYSCALE
    # ---------------------------------------------------------
    gray = cv2.cvtColor(img_shadow_removed, cv2.COLOR_BGR2GRAY)

    fig3, log3 = generate_stage_visuals(gray, 3, "Grayscale")
    outputs.extend([gray, fig3, log3])

    # ---------------------------------------------------------
    # TAHAP 4: BRIGHTNESS
    # ---------------------------------------------------------
    bright = cv2.convertScaleAbs(gray, alpha=1, beta=int(val_bright))

    fig4, log4 = generate_stage_visuals(bright, 4, "Brightness")
    outputs.extend([bright, fig4, log4])

    # ---------------------------------------------------------
    # TAHAP 5: CONTRAST
    # ---------------------------------------------------------
    faktor_c = max(0.1, float(val_contrast) / 50.0)
    contrast = np.clip((bright.astype(np.float32) - 128) * faktor_c + 128, 0, 255).astype(np.uint8)

    fig5, log5 = generate_stage_visuals(contrast, 5, "Contrast")
    outputs.extend([contrast, fig5, log5])

    # ---------------------------------------------------------
    # TAHAP 6: MEDIAN FILTER
    # ---------------------------------------------------------
    k_size = int(val_blur) if int(val_blur) % 2 != 0 else int(val_blur) + 1
    median_filtered = cv2.medianBlur(contrast, k_size)

    fig6, log6 = generate_stage_visuals(median_filtered, 6, "Median Filter")
    outputs.extend([median_filtered, fig6, log6])

    # ---------------------------------------------------------
    # TAHAP 7: THRESHOLDING
    # ---------------------------------------------------------
    t_size = int(val_t_block) if int(val_t_block) % 2 != 0 else int(val_t_block) + 1
    thresh = cv2.adaptiveThreshold(median_filtered, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, t_size, int(val_t_c))

    fig7, log7 = generate_stage_visuals(thresh, 7, "Thresholding")
    outputs.extend([thresh, fig7, log7])

    # ---------------------------------------------------------
    # TAHAP 8: MORPHOLOGY CLOSING (Menyambung Pinggiran Koin yang Putus)
    # ---------------------------------------------------------
    # Menggunakan kernel elips ukuran 15x15 untuk menyambungkan gap pada rim koin
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    morph_closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_close)

    # Melakukan hole filling: Menggambar seluruh kontur luar secara solid
    # untuk memastikan bagian dalam koin menyatu utuh dan tidak bolong-bolong
    temp_contours, _ = cv2.findContours(morph_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    morph_filled = np.zeros_like(morph_closed)
    cv2.drawContours(morph_filled, temp_contours, -1, 255, -1)

    # Tetap kirim visualisasi ke output Gradio dengan nama label asli agar UI sinkron
    fig8, log8 = generate_stage_visuals(morph_filled, 8, "Morphology Opening")
    outputs.extend([morph_filled, fig8, log8])

    # ---------------------------------------------------------
    # TAHAP 9: MORPHOLOGY OPENING (Smoothing & Pembersihan Noise)
    # ---------------------------------------------------------
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    morph_final = cv2.morphologyEx(morph_filled, cv2.MORPH_OPEN, kernel_open)

    fig9, log9 = generate_stage_visuals(morph_final, 9, "Morphology Closing")
    outputs.extend([morph_final, fig9, log9])

    # ---------------------------------------------------------
    # TAHAP 10: CONTOUR DETECTION
    # ---------------------------------------------------------
    contours, _ = cv2.findContours(morph_final.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_contours = img_scaled.copy()
    cv2.drawContours(img_contours, contours, -1, (0, 255, 0), 2)

    fig10, log10 = generate_stage_visuals(morph_final, 10, "Contour Detection")
    outputs.extend([img_contours, fig10, log10])

    # ---------------------------------------------------------
    # TAHAP 11: FEATURE EXTRACTION & CIRCULARITY FILTERING (Via Convex Hull)
    # ---------------------------------------------------------
    valid_contours = []
    tabel_fitur = []

    # Kumpulkan data dasar pixel koin terlebih dahulu
    for c in contours:
        hull = cv2.convexHull(c)
        area = cv2.contourArea(hull)
        if area < 3000:
            continue

        perimeter = cv2.arcLength(hull, True)
        circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

        # Toleransi circularity disesuaikan agar koin yang sedikit lonjong atau miring tetap terdeteksi
        if 0.70 <= circularity <= 1.25:
            ((cx, cy), radius) = cv2.minEnclosingCircle(hull)
            diameter_px  = radius * 2
            x, y, w, h = cv2.boundingRect(hull)
            aspect_ratio = float(w) / h
            solidity = 1.0

            mask = np.zeros(gray.shape, dtype=np.uint8)
            cv2.drawContours(mask, [hull], -1, 255, -1)
            mean_intensity = cv2.mean(gray, mask=mask)[0]

            hu = hitung_hu_moments_log(c)

            fitur = {
                "Diameter": diameter_px,
                "Radius": radius,
                "Area": area,
                "Perimeter": perimeter,
                "Circularity": circularity,
                "Aspect_Ratio": aspect_ratio,
                "Solidity": solidity,
                "Mean_Intensity": mean_intensity,
                "Hu1": hu[0],
                "Contour_Raw": c
            }
            valid_contours.append(fitur)

    # ── METODE AUTOMATIC PPM CALIBRATION (GRID SEARCH OPTIMIZATION) ────────
    # Jika diaktifkan dan ada minimal 2 koin terdeteksi, kita lakukan pencarian
    # PPM terbaik secara matematis yang meminimalkan total error kuadrat terhadap
    # dimensi fisik koin Bank Indonesia (22.0, 23.0, 24.0, 27.0 mm).
    # Ini secara otomatis menyerap efek penggelembungan tepi (dilation) dari morfologi!
    # ────────────────────────────────────────────────────────────────────────
    ppm = float(val_ppm) if float(val_ppm) > 0 else 1.0
    auto_calibrated = False

    if val_auto_ppm and len(valid_contours) >= 2:
        best_ppm = ppm
        min_total_error = float('inf')
        # Spesifikasi fisik BI asli (Kakatua 100=23.0mm, Angklung 1000=24.15mm, Jalak Bali 200=25.0mm, Melati 500=27.0mm)
        nominal_bi_diameters = [23.0, 25.0, 27.0, 24.15]

        # Lakukan sweep nilai PPM dari 4.0 hingga 15.0 px/mm dengan step 0.02
        for test_ppm in np.arange(4.0, 15.0, 0.02):
            total_error = 0.0
            for item in valid_contours:
                d_mm = item["Diameter"] / test_ppm
                # Cari nominal terdekat yang paling logis untuk koin ini
                closest_bi = min(nominal_bi_diameters, key=lambda x: abs(d_mm - x))
                total_error += (d_mm - closest_bi) ** 2

            if total_error < min_total_error:
                min_total_error = total_error
                best_ppm = test_ppm

        ppm = best_ppm
        auto_calibrated = True

    # Hitung ulang dimensi fisik mm menggunakan PPM final & bangun tabel fitur
    for i, item in enumerate(valid_contours):
        item["Diameter_mm"]  = item["Diameter"] / ppm
        item["Area_mm2"]     = item["Area"] / (ppm ** 2)
        item["Perimeter_mm"] = item["Perimeter"] / ppm

        tabel_fitur.append([
            f"Koin #{i+1}",
            f"{item['Area']:.1f}",
            f"{item['Perimeter']:.1f}",
            f"{item['Circularity']:.3f}",
            f"{item['Diameter']:.1f}",
            f"{item['Diameter_mm']:.2f}",
            f"{item['Area_mm2']:.2f}",
            f"{item['Perimeter_mm']:.2f}"
        ])

    outputs.append(tabel_fitur)

    # ---------------------------------------------------------
    # TAHAP 12: RULE BASED CLASSIFICATION (dimensi fisik mm vs. BI)
    # ---------------------------------------------------------
    log_klasifikasi  = "==========================================================\n"
    log_klasifikasi += "HASIL RULE BASED CLASSIFICATION (WEIGHTED SCORING - mm)\n"
    log_klasifikasi += "Referensi: Dimensi Fisik Resmi Bank Indonesia\n"
    log_klasifikasi += "==========================================================\n\n"
    log_klasifikasi += f"[Kalibrasi PPM = {float(val_ppm):.2f} px/mm]\n"
    log_klasifikasi += f"{'Nominal':<10} | {'Diameter BI':>12} | {'Area BI':>12} | {'Perimeter BI':>14}\n"
    log_klasifikasi += f"{'-'*54}\n"
    for nm, rv in REFERENSI_KOIN.items():
        log_klasifikasi += (f"{nm:<10} | {rv['Diameter_mm']:>9.2f} mm | "
                            f"{rv['Area_mm2']:>9.2f} mm² | "
                            f"{rv['Perimeter_mm']:>11.2f} mm\n")
    log_klasifikasi += f"{'='*54}\n\n"

    img_visualisasi = img_scaled.copy()
    koin_count = 0

    for item in valid_contours:
        koin_count += 1
        skor_nominal = {}

        # ── Gaussian Weighted Scoring (mm vs. standar BI) ───────────────────
        # Gaussian: score = exp(-0.5 * ((x - ref) / sigma)^2)
        # Jauh lebih sensitif terhadap perbedaan kecil antar nominal
        # dibanding similarity linear (1 - error_relatif).
        for nominal, ref_val in REFERENSI_KOIN.items():
            skor_total = 0.0
            for fitur_nama, bobot in BOBOT_FITUR.items():
                val_ekstrak   = item[fitur_nama]        # mm / mm² / rasio
                val_referensi = ref_val[fitur_nama]     # standar BI
                sigma         = SIGMA_FITUR[fitur_nama] # toleransi per fitur
                delta         = val_ekstrak - val_referensi
                # Gaussian similarity: 1.0 saat tepat, turun cepat saat meleset
                similarity    = math.exp(-0.5 * (delta / sigma) ** 2)
                skor_total   += similarity * bobot
            skor_nominal[nominal] = skor_total
        # ────────────────────────────────────────────────────────────────────

        hasil_prediksi = max(skor_nominal, key=skor_nominal.get)
        skor_terbaik   = skor_nominal[hasil_prediksi]

        # Jika confidence di bawah threshold → tandai tidak dikenal
        if skor_terbaik < KONFIDENSI_MIN:
            label_tampil = "?"
        else:
            label_tampil = hasil_prediksi

        log_klasifikasi += f"KOIN #{koin_count}\n"
        log_klasifikasi += f"  Diameter Terdeteksi : {item['Diameter']:.1f} px  →  {item['Diameter_mm']:.2f} mm\n"
        log_klasifikasi += f"  Area Terdeteksi     : {item['Area']:.1f} px²  →  {item['Area_mm2']:.2f} mm²\n"
        log_klasifikasi += f"  Perimeter Terdeteksi: {item['Perimeter']:.1f} px  →  {item['Perimeter_mm']:.2f} mm\n"
        log_klasifikasi += f"  Circularity         : {item['Circularity']:.3f}\n"
        log_klasifikasi += f"  Skor Gaussian Similarity vs. Standar BI:\n"
        for nm in REFERENSI_KOIN:
            bintang = " ◀ TERPILIH" if nm == hasil_prediksi else ""
            log_klasifikasi += f"    {nm:<8}: {skor_nominal[nm]:.4f}{bintang}\n"
        log_klasifikasi += f"  Confidence Terbaik : {skor_terbaik:.4f} "
        log_klasifikasi += f"({'OK' if skor_terbaik >= KONFIDENSI_MIN else 'RENDAH - cek PPM!'})\n"
        log_klasifikasi += f"  Hasil Klasifikasi  => {label_tampil}\n"
        log_klasifikasi += f"----------------------------------------------------------\n"

        # TAHAP FINAL: DRAW LABELS
        c_raw = item["Contour_Raw"]
        radius = item["Radius"]
        M = cv2.moments(c_raw)
        if M["m00"] != 0:
            cx_koin = int(M["m10"] / M["m00"])
            cy_koin = int(M["m01"] / M["m00"])
        else:
            ((cx_koin, cy_koin), _) = cv2.minEnclosingCircle(c_raw)
            cx_koin, cy_koin = int(cx_koin), int(cy_koin)

        cv2.circle(img_visualisasi, (cx_koin, cy_koin), int(radius), (0, 255, 0), 4)
        cv2.circle(img_visualisasi, (cx_koin, cy_koin), 5, (255, 0, 0), -1)

        # Label nominal + confidence pada gambar output
        cv2.putText(img_visualisasi, str(label_tampil),
                    (int(cx_koin - radius), int(cy_koin - radius - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 3)

    outputs.append(log_klasifikasi)

    # ---------------------------------------------------------
    # OUTPUT FINAL
    # ---------------------------------------------------------
    outputs.append(img_visualisasi)
    status_sistem = f"🟢 Sukses memproses {koin_count} koin. PPM yang digunakan: {ppm:.3f} px/mm (Auto-Kalibrasi: {'Aktif' if val_auto_ppm and auto_calibrated else 'Mati'})."
    outputs.append(status_sistem)

    return tuple(outputs)

# ==============================================================================
# DESAIN ANTARMUKA GRADIO BLOCKS
# ==============================================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🪙 SISTEM PIPELINE PENGOLAHAN CITRA DIGITAL (DETEKSI KOIN)")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Panel Media Input")
            ui_input_img = gr.Image(type="numpy", label="Foto Koin Utama")

            gr.Markdown("### 🎛️ Konfigurasi Variabel Spasial")
            ui_bright = gr.Slider(-100, 100, value=0, label="Konstanta Brightness")
            ui_contrast = gr.Slider(1, 200, value=50, label="Contrast Alpha")
            ui_blur = gr.Slider(3, 31, value=11, step=2, label="Mean Filter Kernel Size")
            ui_t_block = gr.Slider(3, 99, value=45, step=2, label="Adaptive Threshold Block Size")
            ui_t_c = gr.Slider(-20, 20, value=7, label="Adaptive Threshold Konstanta C")
            ui_ppm = gr.Slider(1.0, 35.0, value=7.00, step=0.05, label="📢 Kalibrasi PPM...")
            ui_auto_ppm = gr.Checkbox(label="💡 Auto-Kalibrasi PPM Dinamis (Sangat Direkomendasikan)", value=True)

            ui_btn = gr.Button("🚀 Analisis Seluruh Pipeline", variant="primary")

    gr.Markdown("---")

    # Penampung Referensi Output UI agar bisa di-mapping ke fungsi
    ui_outputs = []

    # Generate UI Layout TAHAP 1 s.d 10 Secara Dinamis
    tahap_names = [
        "Tahap 1 : Scaling", "Tahap 2 : Shadow Removal", "Tahap 3 : Grayscale",
        "Tahap 4 : Brightness", "Tahap 5 : Contrast", "Tahap 6 : Median Filter",
        "Tahap 7 : Thresholding", "Tahap 8 : Morphology Opening",
        "Tahap 9 : Morphology Closing", "Tahap 10: Contour Detection"
    ]

    for i, name in enumerate(tahap_names):
        gr.Markdown(f"### {name}")
        with gr.Row():
            img_out = gr.Image(label=f"Visualisasi {name}")
            hist_out = gr.Plot(label=f"Histogram {name}")
            log_out = gr.Textbox(label=f"Matriks Piksel {name}", lines=11)
            ui_outputs.extend([img_out, hist_out, log_out])

    gr.Markdown("---")

    # UI TAHAP 11
    gr.Markdown("### Tahap 11 : Feature Extraction")
    ui_table_fe = gr.Dataframe(
        headers=["ID", "Area(px²)", "Perimeter(px)", "Circularity",
                 "Diameter(px)", "Diameter(mm)", "Area(mm²)", "Perimeter(mm)"],
        datatype=["str", "str", "str", "str", "str", "str", "str", "str"],
        label="Tabel Ekstraksi Fitur (pixel & mm)"
    )
    ui_outputs.append(ui_table_fe)

    gr.Markdown("---")

    # UI TAHAP 12
    gr.Markdown("### Tahap 12 : Rule Based Classification")
    ui_log_rulebased = gr.Textbox(label="Log Klasifikasi Rule Based", lines=10)
    ui_outputs.append(ui_log_rulebased)

    gr.Markdown("---")

    # OUTPUT FINAL
    gr.Markdown("### 🖼️ Hasil Akhir Identifikasi (Output Final)")
    with gr.Row():
        ui_out_img_final = gr.Image(label="Output Klasifikasi Nominal Koin")
        ui_status_final = gr.Textbox(label="Status Eksekusi", interactive=False)
    ui_outputs.extend([ui_out_img_final, ui_status_final])

    ui_btn.click(
        fn=pipeline_koin_stabil,
        inputs=[ui_input_img, ui_bright, ui_contrast, ui_blur, ui_t_block, ui_t_c, ui_ppm, ui_auto_ppm],
        outputs=ui_outputs
    )

demo.launch(debug=True)

/tmp/ipykernel_2633/2832734118.py:470: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-5 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

OSError: Cannot find empty port in range: 7860-7959. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.